# scGenePT perturbation-prediction evaluation

**Run:** `scGenePT_adamson_Jul11-2155`  
**Dataset:** Adamson GBM radiotherapy timepoints (`ctrl`, `IR_24h`, `IR_1week`)

This notebook evaluates the validation-selected scGenePT model from its saved held-out
predictions. It mirrors the diagnostics in `Model Evaluation v2.ipynb`, while adapting
them to the actual scGenePT output contract:

- `test_results.npz` contains predictions only for treated test cells;
- each treated cell occurs five times because the graph builder sampled five control
  inputs per cell;
- the training-control mean is the reference used for perturbation deltas;
- the final two variables (`IR_24h`, `IR_1week`) are perturbation flags, not biological
  genes, and are removed from reported gene-level metrics.

The notebook uses the saved arrays and does **not** reload the 1.6 GB model checkpoint.
This makes the evaluation deterministic, lightweight, and independent of the original
scGenePT/GEARS runtime. The `full_retrained` checkpoint is described for provenance only:
it was trained on train + validation + test after held-out evaluation and must not be used
for reported test performance.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)

RUN_NAME = "scGenePT_adamson_Jul11-2155"
default_project = Path("/Users/zhehaowang/PycharmProjects/scgpt_test")
requested_project = Path(os.environ.get("SCGENEPT_PROJECT_DIR", Path.cwd())).expanduser()

project_candidates = [
    requested_project,
    requested_project.parent,
    default_project,
]
PROJECT_DIR = next(
    (path.resolve() for path in project_candidates if (path / RUN_NAME).is_dir()),
    default_project,
)
run_candidates = [PROJECT_DIR / RUN_NAME, PROJECT_DIR / "save_scgenept" / RUN_NAME]
RUN_DIR = next((path for path in run_candidates if path.is_dir()), None)
if RUN_DIR is None:
    raise FileNotFoundError(
        f"Could not find {RUN_NAME}. Checked: " + ", ".join(map(str, run_candidates))
    )

OUTPUT_DIR = Path(
    os.environ.get(
        "SCGENEPT_OUTPUT_DIR",
        PROJECT_DIR / "analysis_outputs" / "scgenept_model_evaluation",
    )
).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_ORDER = ["train", "val", "test"]
CONDITION_ORDER = ["ctrl", "IR_24h", "IR_1week"]
TREATMENTS = ["IR_24h", "IR_1week"]
DISPLAY_CONDITION = {
    "ctrl": "Control",
    "IR_24h": "IR-24h",
    "IR_1week": "IR-1week",
    "average": "Average",
}
CONDITION_COLORS = {
    "ctrl": "#355C7D",
    "IR_24h": "#E76F51",
    "IR_1week": "#2A9D8F",
    "average": "#6C6C6C",
}
SPLIT_COLORS = {"train": "#4C78A8", "val": "#F58518", "test": "#54A24B"}

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "figure.dpi": 120,
        "savefig.dpi": 300,
    }
)

FIGURE_PATHS = {}


def pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    if keep.sum() < 2 or np.std(x[keep]) == 0 or np.std(y[keep]) == 0:
        return float("nan")
    return float(np.corrcoef(x[keep], y[keep])[0, 1])


def save_figure(fig, stem):
    png_path = OUTPUT_DIR / f"{stem}.png"
    pdf_path = OUTPUT_DIR / f"{stem}.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    FIGURE_PATHS[stem] = {"png": str(png_path), "pdf": str(pdf_path)}
    return png_path


print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory:     {RUN_DIR}")
print(f"Output directory:  {OUTPUT_DIR}")

## 1. Load the saved run

Expression arrays are memory-mapped. The only large calculation here is the mean of the
training control cells, which reproduces the baseline used by the training script.

In [ ]:
required_files = [
    "gene_names.json",
    "train_expression.npy",
    "val_expression.npy",
    "test_expression.npy",
    "train_metadata.csv",
    "val_metadata.csv",
    "test_metadata.csv",
    "test_results.npz",
    "test_metrics_full.json",
    "run.log",
]
missing_files = [name for name in required_files if not (RUN_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required run files: {missing_files}")

with (RUN_DIR / "gene_names.json").open(encoding="utf-8") as handle:
    gene_names_full = np.asarray(json.load(handle), dtype=object)
with (RUN_DIR / "test_metrics_full.json").open(encoding="utf-8") as handle:
    saved_test_metrics_json = json.load(handle)

metadata = {}
expression = {}
for split in SPLIT_ORDER:
    metadata[split] = pd.read_csv(RUN_DIR / f"{split}_metadata.csv", index_col=0)
    metadata[split]["condition"] = metadata[split]["condition"].astype(str).str.replace("-", "_", regex=False)
    expression[split] = np.load(RUN_DIR / f"{split}_expression.npy", mmap_mode="r")
    if len(metadata[split]) != expression[split].shape[0]:
        raise ValueError(f"{split}: metadata and expression row counts do not match")

result_file = np.load(RUN_DIR / "test_results.npz", allow_pickle=False)
pred_graph_full = result_file["pred"]
truth_graph_full = result_file["truth"]
pert_graph = np.asarray(
    [str(value).replace("-", "_") for value in result_file["pert_cat"]]
)

mask_path = RUN_DIR / "full_retrained" / "biological_gene_mask.npy"
dummy_path = RUN_DIR / "full_retrained" / "dummy_genes.json"
if dummy_path.exists():
    dummy_genes = json.loads(dummy_path.read_text(encoding="utf-8"))
else:
    dummy_genes = TREATMENTS
if mask_path.exists():
    biological_mask = np.load(mask_path).astype(bool)
else:
    biological_mask = ~np.isin(gene_names_full, dummy_genes)

if len(biological_mask) != len(gene_names_full):
    raise ValueError("Biological mask and gene-name axis do not match")

biological_indices = np.flatnonzero(biological_mask)
gene_names = gene_names_full[biological_mask].astype(str)
duplicate_counts = Counter(gene_names)
feature_labels = np.asarray(
    [
        name if duplicate_counts[name] == 1 else f"{name} [feature {feature_idx}]"
        for name, feature_idx in zip(gene_names, biological_indices)
    ],
    dtype=object,
)

train_ctrl_mask = metadata["train"]["condition"].eq("ctrl").to_numpy()
# Keep NumPy's float32 mean behaviour to match the training script exactly.
ctrl_mean_full = expression["train"][train_ctrl_mask].mean(axis=0)
ctrl_mean = np.asarray(ctrl_mean_full[biological_mask], dtype=np.float32)

asset_summary = pd.DataFrame(
    [
        {"asset": f"{split}_expression", "shape": str(expression[split].shape), "dtype": str(expression[split].dtype)}
        for split in SPLIT_ORDER
    ]
    + [
        {"asset": "test_results/pred", "shape": str(pred_graph_full.shape), "dtype": str(pred_graph_full.dtype)},
        {"asset": "test_results/truth", "shape": str(truth_graph_full.shape), "dtype": str(truth_graph_full.dtype)},
        {"asset": "test_results/pert_cat", "shape": str(pert_graph.shape), "dtype": str(pert_graph.dtype)},
        {"asset": "biological_gene_axis", "shape": str(gene_names.shape), "dtype": str(gene_names.dtype)},
    ]
)
display(asset_summary)

## 2. Split composition and integrity checks

The test prediction file is expanded by graph sampling. For each treated test cell, the
graph builder paired the target with five sampled control inputs. Therefore the correct
expected row count for each condition is `number of treated test cells × 5`, not the full
test-set size.

In [ ]:
split_summary_rows = []
for split in SPLIT_ORDER:
    df = metadata[split]
    split_summary_rows.append(
        {
            "split": split,
            "cells": len(df),
            "percent_total": 100 * len(df) / sum(len(metadata[s]) for s in SPLIT_ORDER),
            "expression_features": expression[split].shape[1],
            "samples": df["Sample"].nunique(),
            "batches": df["Batch"].nunique(),
        }
    )
split_summary = pd.DataFrame(split_summary_rows)

all_metadata = pd.concat(
    [metadata[split].assign(saved_split=split) for split in SPLIT_ORDER],
    axis=0,
)
split_condition_counts = (
    pd.crosstab(all_metadata["saved_split"], all_metadata["condition"])
    .reindex(index=SPLIT_ORDER, columns=CONDITION_ORDER)
    .fillna(0)
    .astype(int)
)
split_condition_percent = split_condition_counts.div(split_condition_counts.sum(axis=1), axis=0) * 100

result_condition_counts = pd.Series(pert_graph).value_counts().reindex(TREATMENTS).astype(int)
test_treatment_counts = metadata["test"]["condition"].value_counts().reindex(TREATMENTS).astype(int)
replicate_ratios = result_condition_counts / test_treatment_counts
if not np.all(np.isfinite(replicate_ratios)) or not np.allclose(replicate_ratios, np.round(replicate_ratios)):
    raise ValueError(f"Could not infer an integer graph replicate count: {replicate_ratios.to_dict()}")
graph_replicates = int(round(float(replicate_ratios.iloc[0])))
if not np.all(replicate_ratios.astype(int) == graph_replicates):
    raise ValueError(f"Graph replicate count differs by condition: {replicate_ratios.to_dict()}")

duplicate_symbol_counts = pd.Series(gene_names).value_counts()
integrity_checks = pd.DataFrame(
    [
        {"check": "pred/truth shapes agree", "value": pred_graph_full.shape == truth_graph_full.shape},
        {"check": "result feature axis matches gene_names.json", "value": pred_graph_full.shape[1] == len(gene_names_full)},
        {"check": "biological features", "value": int(biological_mask.sum())},
        {"check": "excluded dummy features", "value": ", ".join(gene_names_full[~biological_mask].astype(str))},
        {"check": "graph samples per treated test cell", "value": graph_replicates},
        {"check": "test graph rows", "value": len(pert_graph)},
        {"check": "expected test graph rows", "value": int((test_treatment_counts * graph_replicates).sum())},
        {"check": "unique biological gene symbols", "value": int(pd.Series(gene_names).nunique())},
        {"check": "duplicated gene symbols", "value": int((duplicate_symbol_counts > 1).sum())},
        {"check": "extra duplicate feature columns", "value": int(len(gene_names) - pd.Series(gene_names).nunique())},
    ]
)

display(split_summary)
display(split_condition_counts)
display(split_condition_percent.round(1))
display(
    pd.DataFrame(
        {
            "test_treated_cells": test_treatment_counts,
            "saved_graph_rows": result_condition_counts,
            "rows_per_cell": replicate_ratios,
        }
    )
)
display(integrity_checks)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.4), constrained_layout=True)

axes[0].bar(
    split_summary["split"],
    split_summary["cells"],
    color=[SPLIT_COLORS[s] for s in split_summary["split"]],
    width=0.62,
)
axes[0].set_title("A  Saved split sizes", loc="left")
axes[0].set_ylabel("Cells")
axes[0].grid(axis="y", color="#E8E8E8")
axes[0].set_axisbelow(True)
for i, row in split_summary.iterrows():
    axes[0].text(i, row["cells"] + 280, f"{int(row['cells']):,}\n{row['percent_total']:.1f}%", ha="center", fontweight="bold")

bottom = np.zeros(len(SPLIT_ORDER))
for condition in CONDITION_ORDER:
    values = split_condition_percent[condition].to_numpy()
    axes[1].bar(
        SPLIT_ORDER,
        values,
        bottom=bottom,
        color=CONDITION_COLORS[condition],
        label=DISPLAY_CONDITION[condition],
        width=0.62,
    )
    for i, (value, base) in enumerate(zip(values, bottom)):
        if value >= 7:
            axes[1].text(i, base + value / 2, f"{value:.1f}%", ha="center", va="center", color="white", fontweight="bold")
    bottom += values
axes[1].set_title("B  Condition balance within split", loc="left")
axes[1].set_ylabel("Cells within split (%)")
axes[1].set_ylim(0, 100)
axes[1].legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))

save_figure(fig, "figure_scgenept_split_composition")
display(fig)
plt.close(fig)

## 3. Validation trajectory and checkpoint selection

`best_model.pt` was selected using the average validation Pearson correlation of the
treatment-minus-control delta. The parser below accepts signed and scientific-notation
numbers; this matters because the run briefly recorded negative validation correlations.

In [ ]:
log_text = (RUN_DIR / "run.log").read_text(encoding="utf-8", errors="replace")
number = r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
simple_pattern = re.compile(
    rf"(IR_24h|IR_1week|AVERAGE)\s+-> Pearson: {number} \| DE: {number} \| Delta: {number} \| DE_Delta: {number}"
)
full_pattern = re.compile(
    rf"(IR_24h|IR_1week|AVERAGE)\s+-> MSE: {number} \| PCC-delta: {number} \| E-dist: {number} \| Wasserstein: {number} \| KL: {number} \| Common-DEGs: {number}"
)

simple_rows = []
full_rows = []
current_epoch = None
in_full_validation = False
for line in log_text.splitlines():
    epoch_match = re.search(r"\[Epoch\s+(\d+) - Validation\]", line)
    if epoch_match:
        current_epoch = int(epoch_match.group(1))
        in_full_validation = False
        continue
    if "Full six metrics on validation set" in line:
        in_full_validation = True
        continue
    simple_match = simple_pattern.search(line)
    if simple_match and current_epoch is not None and not in_full_validation:
        condition, p, p_de, p_delta, p_delta_de = simple_match.groups()
        simple_rows.append(
            {
                "epoch": current_epoch,
                "condition": condition.lower() if condition == "AVERAGE" else condition,
                "Pearson": float(p),
                "Pearson_DE": float(p_de),
                "Pearson_delta": float(p_delta),
                "Pearson_delta_DE": float(p_delta_de),
            }
        )
    full_match = full_pattern.search(line)
    if full_match and current_epoch is not None and in_full_validation:
        condition, mse, pcc_delta, e_dist, wasserstein, kl, common_degs = full_match.groups()
        full_rows.append(
            {
                "epoch": current_epoch,
                "condition": condition.lower() if condition == "AVERAGE" else condition,
                "MSE": float(mse),
                "PCC_delta": float(pcc_delta),
                "E_distance": float(e_dist),
                "Wasserstein": float(wasserstein),
                "KL_divergence": float(kl),
                "Common_DEGs": float(common_degs),
            }
        )

validation_simple = pd.DataFrame(simple_rows)
validation_full = pd.DataFrame(full_rows)
expected_simple_rows = 60 * 3
if len(validation_simple) != expected_simple_rows:
    print(f"Warning: parsed {len(validation_simple)} simple validation rows; expected {expected_simple_rows}.")
average_validation = validation_simple[validation_simple["condition"].eq("average")]
best_validation = average_validation.loc[[average_validation["Pearson_delta"].idxmax()]].copy()

display(best_validation)
display(validation_simple.tail(9))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.8, 7.6), constrained_layout=True)
plot_conditions = ["IR_24h", "IR_1week", "average"]
styles = {"IR_24h": "-", "IR_1week": "-", "average": "--"}

for condition in plot_conditions:
    df = validation_simple[validation_simple["condition"].eq(condition)]
    label = DISPLAY_CONDITION[condition]
    color = CONDITION_COLORS[condition]
    axes[0, 0].plot(df["epoch"], df["Pearson"], styles[condition], color=color, label=label, linewidth=1.8)
    axes[0, 1].plot(df["epoch"], df["Pearson_delta"], styles[condition], color=color, label=label, linewidth=1.8)

axes[0, 0].set_title("A  Absolute-expression Pearson", loc="left")
axes[0, 1].set_title("B  Delta Pearson (selection metric)", loc="left")
best_epoch = int(best_validation.iloc[0]["epoch"])
best_score = float(best_validation.iloc[0]["Pearson_delta"])
axes[0, 1].scatter([best_epoch], [best_score], marker="*", s=130, color="#111111", zorder=6)
axes[0, 1].annotate(f"best: epoch {best_epoch}\n{best_score:.4f}", (best_epoch, best_score), xytext=(-58, -38), textcoords="offset points")

for condition in plot_conditions:
    df = validation_full[validation_full["condition"].eq(condition)]
    label = DISPLAY_CONDITION[condition]
    color = CONDITION_COLORS[condition]
    axes[1, 0].plot(df["epoch"], df["MSE"], styles[condition], color=color, label=label, linewidth=1.6)
    axes[1, 1].plot(df["epoch"], df["Common_DEGs"], styles[condition], color=color, label=label, linewidth=1.6)

axes[1, 0].set_title("C  Pseudobulk MSE", loc="left")
axes[1, 1].set_title("D  Common DEGs@100", loc="left")
for ax in axes.flat:
    ax.set_xlabel("Epoch")
    ax.grid(color="#E8E8E8")
    ax.set_axisbelow(True)
axes[0, 0].set_ylabel("Pearson r")
axes[0, 1].set_ylabel("Pearson r")
axes[1, 0].set_ylabel("MSE")
axes[1, 1].set_ylabel("Overlap fraction")
axes[0, 0].legend(frameon=False, ncol=3, loc="lower right")

save_figure(fig, "figure_scgenept_validation_trajectory")
display(fig)
plt.close(fig)

## 4. Saved held-out test metrics

These six metrics were produced by the training script after restoring the
validation-selected checkpoint. MSE is between condition-level mean profiles;
PCC-delta is the correlation between predicted and observed mean changes over the
training-control mean; Common-DEGs is top-100 absolute-delta overlap. Distributional
metrics use the expanded graph rows.

In [ ]:
saved_test_metrics = (
    pd.DataFrame.from_dict(saved_test_metrics_json, orient="index")
    .rename_axis("condition")
    .reset_index()
)
saved_test_metrics["condition"] = saved_test_metrics["condition"].str.replace("-", "_", regex=False)
saved_test_metrics["display_condition"] = saved_test_metrics["condition"].map(DISPLAY_CONDITION)
saved_test_metrics = saved_test_metrics.set_index("condition").reindex(TREATMENTS + ["average"]).reset_index()
display(saved_test_metrics)

fig, axes = plt.subplots(2, 2, figsize=(12.4, 8.8))
x = np.arange(len(saved_test_metrics))
labels = saved_test_metrics["display_condition"]
colors = [CONDITION_COLORS[c] for c in saved_test_metrics["condition"]]

width = 0.34
pcc_bars = axes[0, 0].bar(x - width / 2, saved_test_metrics["PCC-delta"], width, label="PCC-delta", color="#4C78A8")
deg_bars = axes[0, 0].bar(x + width / 2, saved_test_metrics["Common-DEGs"], width, label="Common-DEGs@100", color="#F58518")
axes[0, 0].set_ylim(0, 1.12)
axes[0, 0].bar_label(pcc_bars, fmt="%.3f", padding=3, fontsize=8)
axes[0, 0].bar_label(deg_bars, fmt="%.3f", padding=3, fontsize=8)
axes[0, 0].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2)
axes[0, 0].set_title("A  Correlation and DEG recovery", loc="left", pad=16)
axes[0, 0].set_ylabel("Score")

mse_bars = axes[0, 1].bar(x, saved_test_metrics["MSE"], color=colors, width=0.62)
axes[0, 1].set_ylim(0, float(saved_test_metrics["MSE"].max()) * 1.24)
axes[0, 1].bar_label(mse_bars, fmt="%.5f", padding=4, fontsize=8)
axes[0, 1].set_title("B  Pseudobulk MSE", loc="left", pad=16)
axes[0, 1].set_ylabel("MSE (lower is better)")

energy_bars = axes[1, 0].bar(x, saved_test_metrics["E-distance"], color=colors, width=0.62)
axes[1, 0].set_ylim(0, float(saved_test_metrics["E-distance"].max()) * 1.18)
axes[1, 0].bar_label(energy_bars, fmt="%.2f", padding=4, fontsize=8)
axes[1, 0].set_title("C  Energy-distance statistic", loc="left", pad=16)
axes[1, 0].set_ylabel("Distance (lower is better)")

wasserstein_bars = axes[1, 1].bar(x - width / 2, saved_test_metrics["Wasserstein"], width, color="#54A24B", label="Wasserstein")
ax_kl = axes[1, 1].twinx()
kl_bars = ax_kl.bar(x + width / 2, saved_test_metrics["KL-divergence"], width, color="#E45756", alpha=0.75, label="KL divergence")
axes[1, 1].set_ylim(0, float(saved_test_metrics["Wasserstein"].max()) * 1.22)
ax_kl.set_ylim(0, float(saved_test_metrics["KL-divergence"].max()) * 1.22)
axes[1, 1].bar_label(wasserstein_bars, fmt="%.3f", padding=4, fontsize=8)
ax_kl.bar_label(kl_bars, fmt="%.2f", padding=4, fontsize=8)
axes[1, 1].set_title("D  Distributional distances", loc="left", pad=16)
axes[1, 1].set_ylabel("Mean Wasserstein")
ax_kl.set_ylabel("Mean KL divergence")
handles1, labels1 = axes[1, 1].get_legend_handles_labels()
handles2, labels2 = ax_kl.get_legend_handles_labels()
axes[1, 1].legend(
    handles1 + handles2,
    labels1 + labels2,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.99),
    ncol=2,
)
ax_kl.grid(False)

for ax in axes.flat:
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.grid(axis="y", color="#E8E8E8")
    ax.set_axisbelow(True)

fig.suptitle(
    "Held-out scGenePT metric summary",
    x=0.02,
    y=0.985,
    ha="left",
    fontsize=16,
    fontweight="bold",
)
fig.subplots_adjust(left=0.08, right=0.92, bottom=0.09, top=0.89, wspace=0.34, hspace=0.48)

save_figure(fig, "figure_scgenept_final_test_metrics")
display(fig)
plt.close(fig)

## 5. Recover one prediction per treated test cell

Consecutive groups of five graph rows share the same target expression profile and
condition. Averaging the five predictions gives a single prediction per biological test
cell. The target profile is then matched exactly to `test_expression.npy`, allowing the
original test metadata (sample, GBM state, survival annotation, and so on) to be restored.

Condition-level pseudobulk means are unchanged by this collapse because every treated cell
has the same number of graph samples.

In [ ]:
n_graph_rows, n_features = pred_graph_full.shape
if n_graph_rows % graph_replicates != 0:
    raise ValueError("Graph-result rows are not divisible by the inferred replicate count")

pred_groups = pred_graph_full.reshape(-1, graph_replicates, n_features)
truth_groups = truth_graph_full.reshape(-1, graph_replicates, n_features)
condition_groups = pert_graph.reshape(-1, graph_replicates)

repeated_truth_ok = bool(np.all(truth_groups == truth_groups[:, :1, :]))
repeated_condition_ok = bool(np.all(condition_groups == condition_groups[:, :1]))
if not repeated_truth_ok or not repeated_condition_ok:
    raise ValueError("Graph rows are not ordered as consecutive per-cell replicate groups")

cell_pred_full = pred_groups.mean(axis=1)
cell_truth_full = truth_groups[:, 0, :]
cell_conditions = condition_groups[:, 0]

test_expression = expression["test"]
test_metadata = metadata["test"].copy()
treated_positions = np.flatnonzero(test_metadata["condition"].isin(TREATMENTS).to_numpy())
row_lookup = defaultdict(list)
for position in treated_positions:
    row_lookup[np.ascontiguousarray(test_expression[position]).tobytes()].append(int(position))

matched_positions = []
used_positions = set()
for target_row, condition in zip(cell_truth_full, cell_conditions):
    key = np.ascontiguousarray(target_row).tobytes()
    candidates = [
        position
        for position in row_lookup.get(key, [])
        if position not in used_positions and test_metadata.iloc[position]["condition"] == condition
    ]
    if len(candidates) != 1:
        raise ValueError(
            f"Expected one unmatched metadata row for a {condition} target, found {len(candidates)}"
        )
    position = candidates[0]
    matched_positions.append(position)
    used_positions.add(position)

matched_metadata = test_metadata.iloc[matched_positions].copy().reset_index(names="cell_id")
matched_truth_ok = bool(
    np.array_equal(cell_truth_full, np.asarray(test_expression[matched_positions]))
)
condition_match_ok = bool(
    np.array_equal(cell_conditions, matched_metadata["condition"].to_numpy())
)

cell_pred = cell_pred_full[:, biological_mask]
cell_truth = cell_truth_full[:, biological_mask]
pred_graph = pred_graph_full[:, biological_mask]
truth_graph = truth_graph_full[:, biological_mask]

replicate_integrity = pd.DataFrame(
    [
        {"check": "truth identical within every replicate group", "value": repeated_truth_ok},
        {"check": "condition identical within every replicate group", "value": repeated_condition_ok},
        {"check": "collapsed truth matches test_expression exactly", "value": matched_truth_ok},
        {"check": "restored metadata condition matches result condition", "value": condition_match_ok},
        {"check": "collapsed treated test cells", "value": len(cell_truth)},
        {"check": "restored unique metadata rows", "value": len(set(matched_positions))},
    ]
)
display(replicate_integrity)
display(matched_metadata["condition"].value_counts().reindex(TREATMENTS).rename("cells").to_frame())
display(matched_metadata.head())

## 6. Reproduce pseudobulk metrics and inspect duplicate symbols

Metrics are first calculated on all 2,465 biological feature columns, matching the
training script. The dataset has duplicated gene symbols, so exported tables always retain
the original feature index. A symbol-collapsed sensitivity analysis averages duplicated
columns before recalculating correlations; it is reported separately and does not replace
the official feature-level metrics.

In [ ]:
def top_overlap(true_delta, pred_delta, k):
    k = min(int(k), len(true_delta))
    true_idx = np.argsort(np.abs(true_delta))[-k:]
    pred_idx = np.argsort(np.abs(pred_delta))[-k:]
    return len(set(true_idx) & set(pred_idx)) / k


pseudobulk_rows = []
reproduction_rows = []
symbol_rows = []
linearity_rows = []
observed_change_rows = []
gene_vector_frames = []

for condition in TREATMENTS:
    mask = cell_conditions == condition
    observed_mean = cell_truth[mask].mean(axis=0)
    predicted_mean = cell_pred[mask].mean(axis=0)
    true_delta = observed_mean - ctrl_mean
    pred_delta = predicted_mean - ctrl_mean
    residual = pred_delta - true_delta
    linear_coef = np.polyfit(true_delta, pred_delta, 1)
    linear_fit = np.polyval(linear_coef, true_delta)
    quadratic_coef = np.polyfit(true_delta, pred_delta, 2)
    quadratic_fit = np.polyval(quadratic_coef, true_delta)
    total_ss = float(np.sum((pred_delta - np.mean(pred_delta)) ** 2))
    linear_r2 = (
        1 - float(np.sum((pred_delta - linear_fit) ** 2)) / total_ss
        if total_ss > 0
        else float("nan")
    )
    quadratic_r2 = (
        1 - float(np.sum((pred_delta - quadratic_fit) ** 2)) / total_ss
        if total_ss > 0
        else float("nan")
    )

    linearity_rows.append(
        {
            "condition": condition,
            "linear_slope": float(linear_coef[0]),
            "linear_intercept": float(linear_coef[1]),
            "linear_R2": linear_r2,
            "quadratic_R2": quadratic_r2,
            "quadratic_R2_gain": quadratic_r2 - linear_r2,
            "fitted_residual_mean": float(np.mean(pred_delta - linear_fit)),
            "fitted_residual_sd": float(np.std(pred_delta - linear_fit)),
        }
    )
    observed_change_rows.append(
        {
            "condition": condition,
            "mean_observed_delta": float(np.mean(true_delta)),
            "sd_observed_delta": float(np.std(true_delta)),
            "upregulated_feature_percent": 100 * float(np.mean(true_delta > 0)),
            "downregulated_feature_percent": 100 * float(np.mean(true_delta < 0)),
        }
    )

    feature_metrics = {
        "condition": condition,
        "n_test_cells": int(mask.sum()),
        "expression_Pearson": pearson(observed_mean, predicted_mean),
        "expression_MSE": float(np.mean((predicted_mean - observed_mean) ** 2)),
        "expression_MAE": float(np.mean(np.abs(predicted_mean - observed_mean))),
        "delta_Pearson": pearson(true_delta, pred_delta),
        "delta_MAE": float(np.mean(np.abs(residual))),
        "delta_bias": float(np.mean(residual)),
        "top20_delta_Pearson": pearson(
            true_delta[np.argsort(np.abs(true_delta))[-20:]],
            pred_delta[np.argsort(np.abs(true_delta))[-20:]],
        ),
        "top50_overlap": top_overlap(true_delta, pred_delta, 50),
        "top100_overlap": top_overlap(true_delta, pred_delta, 100),
    }
    pseudobulk_rows.append(feature_metrics)

    saved = saved_test_metrics.set_index("condition").loc[condition]
    reproduction_rows.append(
        {
            "condition": condition,
            "saved_MSE": float(saved["MSE"]),
            "recomputed_MSE": feature_metrics["expression_MSE"],
            "absolute_MSE_difference": abs(float(saved["MSE"]) - feature_metrics["expression_MSE"]),
            "saved_PCC_delta": float(saved["PCC-delta"]),
            "recomputed_PCC_delta": feature_metrics["delta_Pearson"],
            "absolute_PCC_difference": abs(float(saved["PCC-delta"]) - feature_metrics["delta_Pearson"]),
            "saved_Common_DEGs_100": float(saved["Common-DEGs"]),
            "recomputed_Common_DEGs_100": feature_metrics["top100_overlap"],
        }
    )

    vectors = pd.DataFrame(
        {
            "condition": condition,
            "feature_index": biological_indices,
            "gene_symbol": gene_names,
            "feature_label": feature_labels,
            "observed_mean": observed_mean,
            "predicted_mean": predicted_mean,
            "true_delta": true_delta,
            "predicted_delta": pred_delta,
            "delta_residual": residual,
        }
    )
    gene_vector_frames.append(vectors)

    symbol_vectors = (
        vectors.groupby("gene_symbol", sort=False)[
            ["observed_mean", "predicted_mean", "true_delta", "predicted_delta"]
        ]
        .mean()
        .reset_index()
    )
    symbol_rows.append(
        {
            "condition": condition,
            "symbols": len(symbol_vectors),
            "expression_Pearson_symbol_collapsed": pearson(symbol_vectors["observed_mean"], symbol_vectors["predicted_mean"]),
            "delta_Pearson_symbol_collapsed": pearson(symbol_vectors["true_delta"], symbol_vectors["predicted_delta"]),
            "delta_MAE_symbol_collapsed": float(np.mean(np.abs(symbol_vectors["predicted_delta"] - symbol_vectors["true_delta"]))),
        }
    )

pseudobulk_metrics = pd.DataFrame(pseudobulk_rows)
metric_reproduction = pd.DataFrame(reproduction_rows)
symbol_sensitivity = pd.DataFrame(symbol_rows)
delta_linearity_diagnostics = pd.DataFrame(linearity_rows)
observed_change_summary = pd.DataFrame(observed_change_rows)
gene_vectors = pd.concat(gene_vector_frames, ignore_index=True)

display(metric_reproduction)
display(pseudobulk_metrics)
display(symbol_sensitivity)
display(delta_linearity_diagnostics)
display(observed_change_summary)

In [ ]:
# Absolute-expression scatter is kept separate from perturbation-delta diagnostics.
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.9), sharex=True, sharey=True)
for col, condition in enumerate(TREATMENTS):
    df = gene_vectors[gene_vectors["condition"].eq(condition)]
    metrics = pseudobulk_metrics.set_index("condition").loc[condition]
    color = CONDITION_COLORS[condition]
    observed = df["observed_mean"].to_numpy()
    predicted = df["predicted_mean"].to_numpy()
    low = float(min(observed.min(), predicted.min()))
    high = float(max(observed.max(), predicted.max()))

    axes[col].scatter(observed, predicted, s=8, alpha=0.32, color=color, edgecolors="none", rasterized=True)
    axes[col].plot([low, high], [low, high], "--", color="#555555", linewidth=1.0)
    axes[col].set_title(f"{DISPLAY_CONDITION[condition]}: observed vs predicted", loc="left", pad=12)
    axes[col].set_xlabel("Observed pseudobulk expression")
    axes[col].text(
        0.04,
        0.94,
        f"r = {metrics['expression_Pearson']:.4f}\nMSE = {metrics['expression_MSE']:.5f}",
        transform=axes[col].transAxes,
        va="top",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#DDDDDD"},
    )
    axes[col].grid(color="#ECECEC")
    axes[col].set_axisbelow(True)
axes[0].set_ylabel("Predicted pseudobulk expression")
fig.suptitle(
    "Observed vs predicted pseudobulk gene-expression levels",
    x=0.02,
    y=0.99,
    ha="left",
    fontsize=15,
    fontweight="bold",
)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.13, top=0.82, wspace=0.18)
save_figure(fig, "figure_scgenept_pseudobulk_expression_scatter")
display(fig)
plt.close(fig)

# A dedicated delta scatter matching the reference visual language.
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.9))
for col, condition in enumerate(TREATMENTS):
    df = gene_vectors[gene_vectors["condition"].eq(condition)]
    metrics = pseudobulk_metrics.set_index("condition").loc[condition]
    linearity = delta_linearity_diagnostics.set_index("condition").loc[condition]
    color = CONDITION_COLORS[condition]
    x_values = df["true_delta"].to_numpy(dtype=float)
    y_values = df["predicted_delta"].to_numpy(dtype=float)
    lim = max(float(np.nanmax(np.abs(np.r_[x_values, y_values]))) * 1.08, 0.10)
    grid = np.linspace(-lim, lim, 200)
    fitted = linearity["linear_slope"] * grid + linearity["linear_intercept"]

    axes[col].scatter(x_values, y_values, s=8, alpha=0.30, color=color, edgecolors="none", rasterized=True)
    axes[col].plot([-lim, lim], [-lim, lim], "--", color="#555555", linewidth=1.0, label="Identity line")
    axes[col].plot(grid, fitted, color="#D55E00", linewidth=1.9, label="Fitted linear trend")
    axes[col].axhline(0, color="#BBBBBB", linewidth=0.8)
    axes[col].axvline(0, color="#BBBBBB", linewidth=0.8)
    axes[col].set_xlim(-lim, lim)
    axes[col].set_ylim(-lim, lim)
    axes[col].set_title(f"{DISPLAY_CONDITION[condition]}: observed vs predicted change", loc="left", pad=12)
    axes[col].set_xlabel("Observed change in expression over train controls")
    axes[col].text(
        0.04,
        0.94,
        (
            f"r = {metrics['delta_Pearson']:.4f}\n"
            f"MAE = {metrics['delta_MAE']:.4f}\n"
            f"slope = {linearity['linear_slope']:.2f}"
        ),
        transform=axes[col].transAxes,
        va="top",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#DDDDDD"},
    )
axes[0].set_ylabel("Predicted change in expression")
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle(
    "Observed vs predicted gene-expression changes",
    x=0.02,
    y=0.99,
    ha="left",
    fontsize=15,
    fontweight="bold",
)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.13, top=0.82, wspace=0.22)
save_figure(fig, "figure_scgenept_delta_scatter")
display(fig)
plt.close(fig)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.8), constrained_layout=True)
quantiles = np.linspace(0.002, 0.998, 400)
for row, condition in enumerate(TREATMENTS):
    df = gene_vectors[gene_vectors["condition"].eq(condition)]
    observed = df["observed_mean"].to_numpy()
    predicted = df["predicted_mean"].to_numpy()
    color = CONDITION_COLORS[condition]
    lo, hi = np.quantile(np.r_[observed, predicted], [0.002, 0.998])
    bins = np.linspace(lo, hi, 70)

    axes[row, 0].hist(observed, bins=bins, density=True, histtype="step", linewidth=1.8, color="#355C7D", label="Observed")
    axes[row, 0].hist(predicted, bins=bins, density=True, histtype="step", linewidth=1.8, color=color, label="Predicted")
    axes[row, 0].set_title(f"{chr(65 + row * 2)}  {DISPLAY_CONDITION[condition]} expression distribution", loc="left")
    axes[row, 0].set_xlabel("Pseudobulk expression across genes")
    axes[row, 0].set_ylabel("Density")
    axes[row, 0].legend(frameon=False)

    q_observed = np.quantile(observed, quantiles)
    q_predicted = np.quantile(predicted, quantiles)
    axes[row, 1].scatter(q_observed, q_predicted, s=9, alpha=0.55, color=color, edgecolors="none")
    q_low = min(q_observed.min(), q_predicted.min())
    q_high = max(q_observed.max(), q_predicted.max())
    axes[row, 1].plot([q_low, q_high], [q_low, q_high], "--", color="#333333", linewidth=1)
    axes[row, 1].set_title(f"{chr(66 + row * 2)}  {DISPLAY_CONDITION[condition]} QQ diagnostic", loc="left")
    axes[row, 1].set_xlabel("Observed expression quantiles")
    axes[row, 1].set_ylabel("Predicted expression quantiles")

for ax in axes.flat:
    ax.grid(color="#ECECEC")
    ax.set_axisbelow(True)

save_figure(fig, "figure_scgenept_expression_distributions_qq")
display(fig)
plt.close(fig)

In [ ]:
from matplotlib.lines import Line2D


# Observed-only change distributions, matching the supplied histogram reference.
all_observed_deltas = gene_vectors["true_delta"].to_numpy(dtype=float)
observed_lim = max(float(np.nanpercentile(np.abs(all_observed_deltas), 99.5)) * 1.08, 0.20)
observed_bins = np.linspace(-observed_lim, observed_lim, 76)
fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.1), sharex=True, sharey=True)
for ax, condition in zip(axes, TREATMENTS):
    values = gene_vectors.loc[gene_vectors["condition"].eq(condition), "true_delta"].to_numpy(dtype=float)
    mean_value = float(np.mean(values))
    up_percent = 100 * float(np.mean(values > 0))
    down_percent = 100 * float(np.mean(values < 0))
    color = CONDITION_COLORS[condition]

    ax.hist(
        values,
        bins=observed_bins,
        density=True,
        color=color,
        alpha=0.36,
        edgecolor=color,
        linewidth=0.8,
    )
    ax.axvline(0, color="#2E7D32", linestyle=":", linewidth=1.5)
    ax.axvline(mean_value, color="#222222", linewidth=1.5)
    ax.set_title(f"{DISPLAY_CONDITION[condition]} vs control", loc="left", pad=14)
    ax.set_xlabel("Observed change in expression over train controls")
    ax.text(
        0.96,
        0.92,
        f"mean = {mean_value:.4f}\nup = {up_percent:.1f}% | down = {down_percent:.1f}%",
        transform=ax.transAxes,
        ha="right",
        va="top",
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#DDDDDD"},
    )
    ax.grid(axis="y", color="#E8E8E8")
    ax.set_axisbelow(True)
axes[0].set_ylabel("Gene density")
legend_handles = [
    Line2D([0], [0], color="#2E7D32", linestyle=":", linewidth=1.5, label="No change"),
    Line2D([0], [0], color="#222222", linewidth=1.5, label="Mean"),
]
fig.legend(handles=legend_handles, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 0.015), ncol=2)
fig.suptitle(
    "Observed gene-expression change distributions after radiotherapy",
    x=0.02,
    y=0.98,
    ha="left",
    fontsize=15,
    fontweight="bold",
)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.23, top=0.80, wspace=0.18)
save_figure(fig, "figure_scgenept_observed_change_distributions")
display(fig)
plt.close(fig)


def binned_residual_summary(x_values, residual_values, n_bins=18):
    order = np.argsort(x_values)
    x_sorted = np.asarray(x_values)[order]
    residual_sorted = np.asarray(residual_values)[order]
    edges = np.linspace(0, len(x_sorted), n_bins + 1, dtype=int)
    rows = []
    for start, end in zip(edges[:-1], edges[1:]):
        if end <= start:
            continue
        x_bin = x_sorted[start:end]
        residual_bin = residual_sorted[start:end]
        rows.append(
            {
                "x_mid": float(np.median(x_bin)),
                "residual_median": float(np.median(residual_bin)),
                "residual_q25": float(np.quantile(residual_bin, 0.25)),
                "residual_q75": float(np.quantile(residual_bin, 0.75)),
            }
        )
    return pd.DataFrame(rows)


# Full reference-style linearity diagnostic: conditions are columns.
fig, axes = plt.subplots(2, 2, figsize=(11.8, 8.2), sharex="col")
linearity_indexed = delta_linearity_diagnostics.set_index("condition")
for col, condition in enumerate(TREATMENTS):
    df = gene_vectors[gene_vectors["condition"].eq(condition)]
    x_values = df["true_delta"].to_numpy(dtype=float)
    y_values = df["predicted_delta"].to_numpy(dtype=float)
    diagnostics = linearity_indexed.loc[condition]
    slope = float(diagnostics["linear_slope"])
    intercept = float(diagnostics["linear_intercept"])
    linear_fit = slope * x_values + intercept
    fitted_residual = y_values - linear_fit
    color = CONDITION_COLORS[condition]

    lim = max(float(np.nanmax(np.abs(np.r_[x_values, y_values]))) * 1.08, 0.10)
    x_line = np.linspace(-lim, lim, 200)
    y_line = slope * x_line + intercept

    scatter_ax = axes[0, col]
    scatter_ax.scatter(x_values, y_values, s=8, alpha=0.30, color=color, edgecolors="none", rasterized=True)
    scatter_ax.plot([-lim, lim], [-lim, lim], color="#555555", linestyle="--", linewidth=1.0, label="Identity line")
    scatter_ax.plot(x_line, y_line, color="#D55E00", linewidth=1.9, label="Fitted linear trend")
    scatter_ax.axhline(0, color="#BBBBBB", linewidth=0.8)
    scatter_ax.axvline(0, color="#BBBBBB", linewidth=0.8)
    scatter_ax.set_xlim(-lim, lim)
    scatter_ax.set_ylim(-lim, lim)
    scatter_ax.set_title(f"A{col + 1}  {DISPLAY_CONDITION[condition]}: observed vs predicted", loc="left", pad=12)
    scatter_ax.set_ylabel("Predicted change in expression")
    scatter_ax.text(
        0.04,
        0.94,
        (
            f"slope = {slope:.2f}\n"
            f"linear R² = {diagnostics['linear_R2']:.3f}\n"
            f"quadratic R² gain = {diagnostics['quadratic_R2_gain']:.3f}"
        ),
        transform=scatter_ax.transAxes,
        va="top",
        fontsize=8.5,
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "#DDDDDD"},
    )

    residual_ax = axes[1, col]
    residual_ax.scatter(x_values, fitted_residual, s=8, alpha=0.24, color=color, edgecolors="none", rasterized=True)
    residual_summary = binned_residual_summary(x_values, fitted_residual)
    residual_ax.fill_between(
        residual_summary["x_mid"],
        residual_summary["residual_q25"],
        residual_summary["residual_q75"],
        color="#222222",
        alpha=0.10,
        linewidth=0,
        label="Binned residual IQR",
    )
    residual_ax.plot(
        residual_summary["x_mid"],
        residual_summary["residual_median"],
        color="#222222",
        linewidth=1.7,
        label="Binned residual median",
    )
    residual_lim = max(float(np.nanpercentile(np.abs(fitted_residual), 99.0)) * 1.25, 0.025)
    residual_ax.axhline(0, color="#555555", linestyle="--", linewidth=1.0)
    residual_ax.set_xlim(-lim, lim)
    residual_ax.set_ylim(-residual_lim, residual_lim)
    residual_ax.set_title(f"B{col + 1}  {DISPLAY_CONDITION[condition]}: residuals from fitted line", loc="left", pad=12)
    residual_ax.set_xlabel("Observed change in expression")
    residual_ax.set_ylabel("Predicted minus fitted linear trend")
    residual_ax.grid(axis="y", color="#E8E8E8")
    residual_ax.set_axisbelow(True)

axes[0, 0].legend(frameon=False, loc="lower right", fontsize=8)
axes[1, 0].legend(frameon=False, loc="lower right", fontsize=8)
fig.suptitle(
    "Linearity diagnostics for observed vs predicted gene-expression changes",
    x=0.02,
    y=0.99,
    ha="left",
    fontsize=15,
    fontweight="bold",
)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.09, top=0.88, wspace=0.25, hspace=0.38)
save_figure(fig, "figure_scgenept_delta_linearity_diagnostics")
display(fig)
plt.close(fig)

## 7. Top radiation-response genes

For each timepoint, the 10 most negative and 10 most positive observed pseudobulk deltas
are selected. Boxplots show the held-out single-cell observed delta distribution relative
to the training-control mean; red points show the mean of the collapsed scGenePT
predictions. Feature labels include the original column index when a symbol is duplicated.

In [ ]:
top_gene_rows = []
top_gene_indices = {}
for condition in TREATMENTS:
    df = gene_vectors[gene_vectors["condition"].eq(condition)].copy()
    order = np.argsort(df["true_delta"].to_numpy())
    selected_local = np.r_[order[:10], order[-10:]]
    selected_df = df.iloc[selected_local].sort_values("true_delta").reset_index(drop=True)
    selected_df["rank"] = np.arange(1, len(selected_df) + 1)
    selected_df["direction"] = np.where(selected_df["true_delta"] < 0, "down", "up")
    top_gene_rows.append(selected_df)
    top_gene_indices[condition] = selected_df["feature_index"].map(
        {full_idx: local_idx for local_idx, full_idx in enumerate(biological_indices)}
    ).to_numpy(dtype=int)

top_genes = pd.concat(top_gene_rows, ignore_index=True)
display(top_genes[["condition", "rank", "feature_index", "gene_symbol", "feature_label", "direction", "true_delta", "predicted_delta"]])

fig, axes = plt.subplots(2, 1, figsize=(15.2, 9.0), constrained_layout=True)
for row, condition in enumerate(TREATMENTS):
    condition_mask = cell_conditions == condition
    local_indices = top_gene_indices[condition]
    df = top_genes[top_genes["condition"].eq(condition)].sort_values("true_delta")
    observed_delta_cells = cell_truth[condition_mask][:, local_indices] - ctrl_mean[local_indices]
    boxplot_data = [observed_delta_cells[:, i] for i in range(observed_delta_cells.shape[1])]
    bp = axes[row].boxplot(boxplot_data, showfliers=False, patch_artist=True, widths=0.52)
    for box in bp["boxes"]:
        box.set_facecolor("white")
        box.set_edgecolor("#333333")
    for median in bp["medians"]:
        median.set_color("#666666")
    axes[row].scatter(
        np.arange(1, len(df) + 1),
        df["predicted_delta"],
        color="#D62728",
        s=34,
        zorder=6,
        label="Predicted mean delta",
    )
    axes[row].axhline(0, color="#2E8B57", linestyle="--", linewidth=1)
    axes[row].set_xticks(np.arange(1, len(df) + 1))
    axes[row].set_xticklabels(df["feature_label"], rotation=55, ha="right", fontsize=9)
    axes[row].set_ylabel("Expression change over train controls")
    axes[row].set_title(f"{chr(65 + row)}  {DISPLAY_CONDITION[condition]}: observed cell distribution and predicted mean", loc="left")
    axes[row].legend(frameon=False)
    axes[row].grid(axis="y", color="#ECECEC")
    axes[row].set_axisbelow(True)

save_figure(fig, "figure_scgenept_top20_deg_boxplots")
display(fig)
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(15.2, 8.2), constrained_layout=True)
for row, condition in enumerate(TREATMENTS):
    df = top_genes[top_genes["condition"].eq(condition)].sort_values("true_delta")
    x = np.arange(len(df))
    width = 0.38
    axes[row].bar(x - width / 2, df["true_delta"], width, label="Observed mean delta", color="#4C78A8")
    axes[row].bar(x + width / 2, df["predicted_delta"], width, label="Predicted mean delta", color="#F58518")
    axes[row].axhline(0, color="#333333", linewidth=0.8)
    axes[row].set_xticks(x)
    axes[row].set_xticklabels(df["feature_label"], rotation=55, ha="right", fontsize=9)
    axes[row].set_ylabel("Mean expression change")
    axes[row].set_title(f"{chr(65 + row)}  {DISPLAY_CONDITION[condition]}: top observed response genes", loc="left")
    axes[row].legend(frameon=False, ncol=2)
    axes[row].grid(axis="y", color="#ECECEC")
    axes[row].set_axisbelow(True)

save_figure(fig, "figure_scgenept_top20_deg_barcharts")
display(fig)
plt.close(fig)

## 8. Cell-level and malignant-state diagnostics

Exact target-to-metadata recovery enables a diagnostic that was not safe in the older
scGPT notebook. Predictions are first averaged across the five control-sampling graphs.
Per-cell correlations are then calculated across biological features and summarised by
broad malignant state. These are descriptive diagnostics, not additional independently
held-out tests, so small groups should be interpreted cautiously.

In [ ]:
def rowwise_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x_centered = x - x.mean(axis=1, keepdims=True)
    y_centered = y - y.mean(axis=1, keepdims=True)
    numerator = np.sum(x_centered * y_centered, axis=1)
    denominator = np.sqrt(np.sum(x_centered**2, axis=1) * np.sum(y_centered**2, axis=1))
    return np.divide(numerator, denominator, out=np.full(len(x), np.nan), where=denominator > 0)


cell_metrics = matched_metadata.copy()
cell_metrics["display_condition"] = cell_metrics["condition"].map(DISPLAY_CONDITION)
cell_metrics["expression_RMSE"] = np.sqrt(np.mean((cell_pred - cell_truth) ** 2, axis=1))
cell_metrics["expression_Pearson"] = rowwise_pearson(cell_truth, cell_pred)
cell_metrics["delta_Pearson"] = rowwise_pearson(cell_truth - ctrl_mean, cell_pred - ctrl_mean)
cell_metrics["delta_MAE"] = np.mean(np.abs((cell_pred - ctrl_mean) - (cell_truth - ctrl_mean)), axis=1)

subgroup_metrics = (
    cell_metrics.groupby(["condition", "GBM_malig_subtypes"], dropna=False)
    .agg(
        n_cells=("cell_id", "size"),
        median_expression_RMSE=("expression_RMSE", "median"),
        median_expression_Pearson=("expression_Pearson", "median"),
        median_delta_Pearson=("delta_Pearson", "median"),
        median_delta_MAE=("delta_MAE", "median"),
    )
    .reset_index()
    .sort_values(["condition", "GBM_malig_subtypes"])
)
display(subgroup_metrics)

state_order = sorted(subgroup_metrics["GBM_malig_subtypes"].dropna().unique())
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8), sharey=True, constrained_layout=True)
for ax, condition in zip(axes, TREATMENTS):
    df = subgroup_metrics[subgroup_metrics["condition"].eq(condition)].set_index("GBM_malig_subtypes").reindex(state_order)
    bars = ax.barh(state_order, df["median_delta_Pearson"], color=CONDITION_COLORS[condition], alpha=0.88)
    ax.set_title(f"{DISPLAY_CONDITION[condition]}: median per-cell delta Pearson", loc="left")
    ax.set_xlabel("Median Pearson r across genes")
    ax.set_xlim(-1, 1)
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.grid(axis="x", color="#ECECEC")
    ax.set_axisbelow(True)
    for bar, n_cells in zip(bars, df["n_cells"]):
        if np.isfinite(n_cells):
            x_text = max(min(bar.get_width() + 0.03, 0.92), -0.92)
            ax.text(x_text, bar.get_y() + bar.get_height() / 2, f"n={int(n_cells)}", va="center", fontsize=9)

save_figure(fig, "figure_scgenept_cell_state_delta_performance")
display(fig)
plt.close(fig)

## 9. Full-retraining provenance — not held-out evaluation

The files under `full_retrained/` document a separate five-epoch continuation trained on
train + validation + test. That checkpoint is intended for downstream GRN or
interpretability work after model selection. Displaying its optimisation history does not
turn it into a valid held-out test model, and none of the performance figures above use it.

In [ ]:
full_dir = RUN_DIR / "full_retrained"
model_config = json.loads((full_dir / "model_config.json").read_text(encoding="utf-8"))
run_args = json.loads((full_dir / "run_args.json").read_text(encoding="utf-8"))
full_history = pd.DataFrame(json.loads((full_dir / "full_retrain_history.json").read_text(encoding="utf-8")))

display(pd.DataFrame({"setting": model_config.keys(), "value": [model_config[key] for key in model_config]}))
display(full_history)

fig, ax = plt.subplots(figsize=(8.8, 4.4), constrained_layout=True)
ax.plot(full_history["epoch"], full_history["loss"], marker="o", linewidth=1.8, label="Total loss")
ax.plot(full_history["epoch"], full_history["mvm_mse"], marker="o", linewidth=1.8, label="MVM MSE")
ax.plot(full_history["epoch"], full_history["response_loss"], marker="o", linewidth=1.8, label="Response loss")
ax.set_xlabel("Full-retraining epoch")
ax.set_ylabel("Training loss")
ax.set_title("Post-test full-data optimisation history (provenance only)", loc="left")
ax.set_xticks(full_history["epoch"])
ax.grid(color="#ECECEC")
ax.set_axisbelow(True)
ax.legend(frameon=False, ncol=3)
ax_lr = ax.twinx()
ax_lr.plot(full_history["epoch"], full_history["lr"], color="#6C6C6C", linestyle="--", marker="s", label="Learning rate")
ax_lr.set_ylabel("Learning rate")

save_figure(fig, "figure_scgenept_full_retrain_history_provenance")
display(fig)
plt.close(fig)

## 10. Export reproducible tables and figures

All tables are written as CSV. Every figure is saved as both PNG and PDF under
`analysis_outputs/scgenept_model_evaluation` unless `SCGENEPT_OUTPUT_DIR` is set.

In [ ]:
tables = {
    "asset_summary": asset_summary,
    "split_summary": split_summary,
    "split_condition_counts": split_condition_counts.reset_index(names="split"),
    "split_condition_percent": split_condition_percent.reset_index(names="split"),
    "integrity_checks": integrity_checks,
    "replicate_integrity": replicate_integrity,
    "validation_simple_metrics": validation_simple,
    "validation_full_metrics": validation_full,
    "best_validation_epoch_by_average_delta": best_validation,
    "saved_test_metrics": saved_test_metrics,
    "metric_reproduction": metric_reproduction,
    "pseudobulk_metrics": pseudobulk_metrics,
    "delta_linearity_diagnostics": delta_linearity_diagnostics,
    "observed_change_summary": observed_change_summary,
    "gene_symbol_collapsed_sensitivity": symbol_sensitivity,
    "gene_level_pseudobulk_values": gene_vectors,
    "top20_response_genes": top_genes,
    "cell_level_metrics_with_metadata": cell_metrics,
    "malignant_state_metrics": subgroup_metrics,
    "full_retrain_history_provenance": full_history,
}

table_paths = {}
for name, table in tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    table_paths[name] = str(path)

manifest = {
    "run_name": RUN_NAME,
    "run_directory": str(RUN_DIR),
    "evaluation_source": "held-out predictions from validation-selected best_model.pt",
    "prediction_file": str(RUN_DIR / "test_results.npz"),
    "excluded_from_performance_evaluation": str(full_dir / "final_model_full_finetuned.pt"),
    "graph_replicates_per_treated_cell": graph_replicates,
    "biological_features": int(biological_mask.sum()),
    "dummy_features_excluded": gene_names_full[~biological_mask].astype(str).tolist(),
    "tables": table_paths,
    "figures": FIGURE_PATHS,
}
manifest_path = OUTPUT_DIR / "evaluation_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

export_summary = pd.DataFrame(
    [
        {"artifact_type": "table", "count": len(table_paths), "directory": str(OUTPUT_DIR)},
        {"artifact_type": "figure (PNG + PDF)", "count": len(FIGURE_PATHS), "directory": str(OUTPUT_DIR)},
        {"artifact_type": "manifest", "count": 1, "directory": str(manifest_path)},
    ]
)
display(export_summary)
print(f"Saved {len(table_paths)} tables, {len(FIGURE_PATHS)} figures in two formats, and one manifest.")

## Evaluation summary

The final cell generates a concise, value-backed interpretation from the notebook's
computed tables. It deliberately distinguishes held-out performance from the post-test
full-data checkpoint.

In [ ]:
average_saved = saved_test_metrics.set_index("condition").loc["average"]
best_row = best_validation.iloc[0]
expression_values = pseudobulk_metrics.set_index("condition")["expression_Pearson"]

summary = f'''
### Main result

- The validation-selected scGenePT checkpoint achieved an average held-out
  **PCC-delta of {average_saved['PCC-delta']:.4f}**, mean-profile
  **MSE of {average_saved['MSE']:.6f}**, and **Common-DEGs@100 of
  {average_saved['Common-DEGs']:.3f}** across IR-24h and IR-1week.
- The best average validation delta correlation in the rounded run log occurred at
  **epoch {int(best_row['epoch'])}** ({best_row['Pearson_delta']:.4f}).
- Absolute pseudobulk expression correlations were
  **{expression_values['IR_24h']:.4f}** for IR-24h and
  **{expression_values['IR_1week']:.4f}** for IR-1week.
- Integrity checks recovered **{len(cell_truth):,} treated test cells** from
  {len(pert_graph):,} graph rows by collapsing **{graph_replicates} control-sampling
  predictions per cell**. The collapsed targets matched `test_expression.npy` exactly.
- Reported performance uses `best_model.pt` / `test_results.npz`. The checkpoint under
  `full_retrained/` used all data, including the test cells, and is restricted to
  downstream GRN/interpretability analysis.

Outputs are available in `{OUTPUT_DIR}`.
'''
display(Markdown(summary))